In [1]:
!git clone https://github.com/unza-speech-lab/zambezi-voice-nyanja.git


Cloning into 'zambezi-voice-nyanja'...
remote: Enumerating objects: 18364, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 18364 (delta 1), reused 8 (delta 1), pack-reused 18345 (from 1)
Receiving objects: 100% (18364/18364), 3.93 GiB | 38.00 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Updating files: 100% (9171/9171), done.


In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets evaluate accelerate
!pip install scikit-learn pandas numpy
!pip install jiwer  # WER metric backend
!pip install huggingface_hub


Looking in indexes: https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.6 MB/s eta 0:00:00


In [3]:
# -*- coding: utf-8 -*-
"""ASR_NYANJA_WHISPER.ipynb"""

import os
import re
import gc
import torch
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List

from datasets import Dataset, Audio
from evaluate import load
from sklearn.model_selection import train_test_split
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)
import transformers

print("Transformers version:", transformers.__version__)
set_seed(42)

# ============================================================
# 2  Paths — dataset, audio, model and output directories
# ============================================================

DATA_DIR   = "/content/zambezi-voice-nyanja/nya/"
AUDIO_DIR  = os.path.join(DATA_DIR, "audio/")
TRAIN_TSV  = os.path.join("/content/zambezi-voice-nyanja/nya/train.tsv")
OUTPUT_DIR = "./nyanja-whisper-model"
FINAL_DIR  = "./nyanja-whisper-final"
MODEL_NAME = "openai/whisper-medium"

PROCESSED_DATA_DIR  = "./processed_data"
PROCESSED_TRAIN_DIR = os.path.join(PROCESSED_DATA_DIR, "train")
PROCESSED_VAL_DIR   = os.path.join(PROCESSED_DATA_DIR, "val")

# ============================================================
# 3  Load Nyanja dataset from CSV/TSV and verify audio files exist
# ============================================================

print("Loading Nyanja dataset...")

df = pd.read_csv(TRAIN_TSV, sep="\t")
df = df[["audio_id", "sentence"]].rename(columns={"sentence": "text"})
df["path"] = AUDIO_DIR + df["audio_id"]
df = df.dropna()

total_before = len(df)
df["exists"]  = df["path"].apply(os.path.exists)
found_df      = df[df["exists"]]
missing_df    = df[~df["exists"]]

print(f"Total rows in TSV (after dropna): {total_before}")
print(f"Audio files FOUND:   {len(found_df)}")
print(f"Audio files MISSING: {len(missing_df)}")

if len(missing_df) > 0:
    print(f"Missing rate: {len(missing_df)/total_before*100:.1f}%")
    for _, row in missing_df.head(10).iterrows():
        print(f"  {row['path']}")

df = df[df["exists"]][["path", "text"]].reset_index(drop=True)
print(f"\nUsable samples: {len(df)}")

# ============================================================
# 4  Train / validation split (90% / 10%)
# ============================================================

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))

# ============================================================
# 5  Convert to HuggingFace Dataset and cast audio to 16 kHz
# ============================================================

def create_dataset(df):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.cast_column("path", Audio(sampling_rate=16000))
    return dataset

train_dataset = create_dataset(train_df)
val_dataset   = create_dataset(val_df)

# ============================================================
# 6  Clean text — lowercase, strip punctuation, digits and non-ASCII (Nyanja)
# ============================================================

CHARS_TO_REMOVE    = r'[,?.!\-;:""%\'"\'\u2018\u2019\u2026()[\]{}]'
NON_NYANJA_LETTERS = r'[qx]'

def clean_text(batch):
    text = batch["text"].lower()
    text = re.sub(CHARS_TO_REMOVE, "", text)
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(NON_NYANJA_LETTERS, "", text)
    text = re.sub(r"[^\x00-\x7F]", "", text)
    text = re.sub(r" +", " ", text).strip()
    batch["text"] = text
    return batch

train_dataset = train_dataset.map(clean_text)
val_dataset   = val_dataset.map(clean_text)

train_dataset = train_dataset.filter(lambda x: len(x["text"].strip()) > 0)
val_dataset   = val_dataset.filter(lambda x: len(x["text"].strip()) > 0)

print("\n--- Sample cleaned text ---")
for i in range(5):
    print(repr(train_dataset[i]["text"]))
print("---\n")

# ============================================================
# 7  Whisper processor — force Shona as phonetic proxy language for Nyanja
# ============================================================
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)

tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language="shona", task="transcribe")
processor  = WhisperProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

# ============================================================
# 8  Prepare dataset — extract features and cache to disk
# ============================================================

def prepare_dataset(batch):
    audio = batch["path"]
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt",
    ).input_features[0]
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch

processed_data_exists = os.path.isdir(PROCESSED_TRAIN_DIR) and os.path.isdir(PROCESSED_VAL_DIR)

if processed_data_exists:
    try:
        print(f"Found cached processed dataset in '{PROCESSED_DATA_DIR}', loading from disk...")
        train_dataset = Dataset.load_from_disk(PROCESSED_TRAIN_DIR)
        val_dataset   = Dataset.load_from_disk(PROCESSED_VAL_DIR)
    except Exception as e:
        print(f"Cached dataset looked incomplete ({e}); recomputing instead.")
        processed_data_exists = False

if not processed_data_exists:
    train_dataset = train_dataset.map(
        prepare_dataset,
        remove_columns=train_dataset.column_names,
        num_proc=1,
        keep_in_memory=False,
        writer_batch_size=50,
    )
    val_dataset = val_dataset.map(
        prepare_dataset,
        remove_columns=val_dataset.column_names,
        num_proc=1,
        keep_in_memory=False,
        writer_batch_size=50,
    )

    print(f"Saving processed datasets to disk at '{PROCESSED_DATA_DIR}'...")
    train_dataset.save_to_disk(PROCESSED_TRAIN_DIR)
    val_dataset.save_to_disk(PROCESSED_VAL_DIR)

    del train_dataset, val_dataset
    gc.collect()

    train_dataset = Dataset.load_from_disk(PROCESSED_TRAIN_DIR)
    val_dataset   = Dataset.load_from_disk(PROCESSED_VAL_DIR)

print("Train after preparation:", len(train_dataset))
print("Validation after preparation:", len(val_dataset))

# ============================================================
# 9  Data collator — pad features and labels into batches
# ============================================================

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)



Transformers version: 5.12.1
Loading Tonga dataset...
Total rows in TSV (after dropna): 8117
Audio files FOUND:   8117
Audio files MISSING: 0

Usable samples: 8117
Train: 7305
Validation: 812


Map:   0%|          | 0/7305 [00:00<?, ? examples/s]

Map:   0%|          | 0/812 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7305 [00:00<?, ? examples/s]

Filter:   0%|          | 0/812 [00:00<?, ? examples/s]


--- Sample cleaned text ---
'mnyamata wina wovala malaya achikasu a manja aatali ndi jinzi atakhala pansi ndi akulu asanu patsogolo pake mmodzi akugwiritsira ntchito kamera ya kanema'
'anyamata asanu ndi atsikana awiri akucheza kukhitchini ndi chakudya'
'opambana one two ndi three pamwambo akuwonetsa chithunzi aliyense atanyamula maluwa'
'mayi wina wa ku asia atavala chovala chaching ono ndi g string akuyang ana maonekedwe ake ataima kumbuyo kwa galasi m chipinda'
'mwamuna mkazi ndi mwamuna wina akuyenda m munda waudzu'
---



Found cached processed dataset in './processed_data', loading from disk...
Train after preparation: 7305
Validation after preparation: 812


In [4]:
# ============================================================
# 10  Load model and set forced decoder language / task
# ============================================================
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

forced_decoder_ids = processor.get_decoder_prompt_ids(language="shona", task="transcribe")
model.generation_config.forced_decoder_ids = forced_decoder_ids
model.generation_config.suppress_tokens    = []
model.generation_config.language           = "shona"
model.generation_config.task               = "transcribe"

print(f"Model: {MODEL_NAME}  |  Parameters: {model.num_parameters():,}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=100,
    save_steps=200,
    save_strategy="steps",
    num_train_epochs=2.1,
    learning_rate=1e-5,
    warmup_steps=300,
    weight_decay=0.01,
    gradient_checkpointing=True,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
)

# ============================================================
# 11  Clear legacy suppress_tokens on model config
# ============================================================
model.config.suppress_tokens = None

# ============================================================
# 12  Word Error Rate (WER) metric for evaluation
# ============================================================
wer_metric = load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 padding before decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# ============================================================
# 13  Build Seq2SeqTrainer
# ============================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

# ============================================================
# 14  Train the model
# ============================================================
print("\n========== STARTING TRAINING ==========")
print(f"Model                : {MODEL_NAME}")
print(f"Transformers version : {transformers.__version__}")
print(f"Train samples        : {len(train_dataset)}")
print(f"Val samples          : {len(val_dataset)}")
print(f"Epochs               : {training_args.num_train_epochs}")
print(f"Effective batch size : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate        : {training_args.learning_rate}")
print("========================================\n")

trainer.train()

# ============================================================
# 15  Save final model and processor
# ============================================================
model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"Model and processor saved to {FINAL_DIR}")

# ============================================================
# 16  Final evaluation on validation set
# ============================================================
print("\n========== FINAL EVALUATION ==========")
eval_results = trainer.evaluate()
print(f"Final WER: {eval_results['eval_wer']:.4f}  ({eval_results['eval_wer']*100:.2f}%)")

# ============================================================
# 17  Spot-check inference on 5 validation samples
# ============================================================
def transcribe(idx):
    example = val_dataset[idx]
    device  = model.device

    input_features = torch.tensor(
        example["input_features"]
    ).unsqueeze(0).to(device)

    infer_forced_ids = processor.get_decoder_prompt_ids(language="shona", task="transcribe")

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=infer_forced_ids,
        )

    prediction = processor.tokenizer.decode(
        predicted_ids[0], skip_special_tokens=True
    )

    label_ids = [i for i in example["labels"] if i != -100]
    reference = processor.tokenizer.decode(
        label_ids, skip_special_tokens=True
    )

    print(f"\n--- Sample {idx} ---")
    print("REFERENCE :", reference)
    print("PREDICTION:", prediction)

for i in range(5):
    transcribe(i)

print("\n========== TRAINING PIPELINE COMPLETE ==========")


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

Model: openai/whisper-medium  |  Parameters: 763,857,920

========== STARTING TRAINING ==========
Model                : openai/whisper-medium
Transformers version : 5.12.1
Train samples        : 7305
Val samples          : 812
Epochs               : 2.1
Effective batch size : 8
Learning rate        : 1e-05



Step,Training Loss,Validation Loss,Wer
200,1.106042,0.378908,0.281634
400,0.603336,0.271468,0.212315
600,0.463217,0.221461,0.174586
800,0.425945,0.194949,0.161417
1000,0.249251,0.182572,0.152518
1200,0.230188,0.175596,0.144510


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and processor saved to ./tonga-whisper-final

========== FINAL EVALUATION ==========


KeyboardInterrupt: 

In [5]:
import os
from huggingface_hub import HfApi, login

# 1. Authenticate with the provided token
HF_TOKEN = os.environ.get("HF_TOKEN")  # set via: export HF_TOKEN=hf_xxx
login(token=HF_TOKEN)

# 2. Define the repository name
repo_id = "Nyanja-whisper-medium-shona-proxy"

print(f"Pushing model from {FINAL_DIR} to Hugging Face Hub: {repo_id}...")

# 3. Push the model and processor
model.push_to_hub(repo_id)
processor.push_to_hub(repo_id)

print(f"Successfully pushed model to https://huggingface.co/{repo_id}")


Pushing model from ./tonga-whisper-final to Hugging Face Hub: Nyanja-whisper-medium-shona-proxy...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...70ggg6m/model.safetensors:   0%|          | 99.6kB / 3.06GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Successfully pushed model to https://huggingface.co/Nyanja-whisper-medium-shona-proxy
